### Import Dependencies

In [14]:
import openai
from qdrant_client import QdrantClient

In [15]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")
openai.api_key = os.getenv("OPENAI_API_KEY")
openai._reset_client()

# optional sanity check — should print True, not show the key
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY loaded: True


### Embedding function

In [16]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval function

In [17]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [18]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [19]:
retrieved_context = retrieve_data("Do you have a USB connectable fan for hot summers.", k=10)

In [20]:
retrieved_context

{'retrieved_context_ids': ['B0BRJS644Z',
  'B0BXC72RLD',
  'B099N9F3FP',
  'B0C8DBH7ZT',
  'B0BM9THPDQ',
  'B0CF57H28T',
  'B0C9QZS95R',
  'B0BYD7PGV1',
  'B09VDLH5M6',
  'B0CC4HBS85'],
 'retrieved_context': ['Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that supp

### Format retrieved context function

In [21]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [22]:
preprocessed_context = process_context(retrieved_context)

In [23]:
print(preprocessed_context)

- ID: B0BRJS644Z, rating: 4.7, description: Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】 This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed Control Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】 Rubber Feet on the fan raise it up enough off of the surface it is sitti

### Create prompt template function

In [24]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [25]:
prompt = build_prompt(preprocessed_context, "Do you have a USB connectable fan for hot summers?")

In [26]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- ID: B0BRJS644Z, rating: 4.7, description: Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】 This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed Control Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility U

### Generate answer function

In [27]:
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content

In [28]:
print(generate_answer(prompt))

Yes. We have two USB-connectable fan options for hot summers:

1) Marame 120mm 5V USB Powered Fan with Speed Controller (ID: B0BRJS644Z)
- USB powered with a 3.3 ft USB cable
- Speed controller on the cord: off/low/medium/high

2) HZD Desk Fan Rechargeable / Mini Portable Fan (note: it’s USB powered, and the listing says it doesn’t include a battery) (ID: B0BXC72RLD)
- USB powered with a 4.9 ft USB cable
- 3 speeds: low/medium/high

If you tell me where you’ll use it (desk, TV box/router, or personal cooling) and whether you need speed control, I can point you to the best match.


### Combined RAG pipeline

In [29]:
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer
    

In [30]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

Yes. We have a USB-powered fan that works well for hot summers:

- Marame 120mm 5V USB Powered Fan (B0BRJS644Z): USB-powered (with a 3.3 ft USB cable) and includes a speed controller to set off/low/medium/high, helping cool routers, TV boxes, modems, and other electronics in warm conditions.


In [31]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

Sure. Here are the earphones in the available products list with a rating above 4.0:

1) B0CH8DRD6K (4.9) – 2 Pack iPhone wired headphones (3.5mm Jack), Apple MFi certified, with microphone and volume control. Compatible with iPhone/iPad/iPod and other 3.5mm devices (adapter may be needed for iPhones without a 3.5mm jack).

2) B0BRV544MV (4.4) – Wekily Bluetooth 5.3 wireless earbuds with a charging case, deep bass, 4-mic clear call, IPX7 waterproof, LED display, and up to 40H total playtime (case + earbuds).

3) B0BM657X74 (4.6) – Bluetooth sleep headband headphones (side-sleeper friendly), with built-in microphone and volume control; listed playtime 10–12 hours; adjustable and washable headband.

If you tell me whether you want wired or Bluetooth (and what device you’ll use them with), I can narrow it down to the best match.


In [32]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))

Here are earphones with a rating below 4 that are available:

1) Pamu S29 Wireless Earbuds (B09TFM1SFQ) – Rating 3.9  
Bluetooth 5.2 earbuds with ANC (noise cancellation), built-in mic/ENC call noise reduction, and a charging case. Battery life up to 30 hours total (with case) and 6 hours per charge. USB-C fast charge.

If you meant “4.0 or lower” (instead of strictly below 4), tell me and I can include the items rated 4.0 exactly (none are shown here, but I’ll re-check if you want that rule).
